In [7]:
import numpy as np
import pandas as pd
import os
import re
import sys
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append(os.path.abspath(os.path.join("..")))
from src.preprocessing.cleaning import apply_cleaning_pipeline

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
DATA = "data"
RAW = "raw"
SOFTWARE_DATA = "software_data"
CONFIGS = "configs"
settings_path = os.path.join("..", CONFIGS, "settings.yaml")
software_path = os.path.join("..", DATA, RAW, SOFTWARE_DATA,  "software.jsonl.gz")
meta_software_path = os.path.join("..", DATA, RAW, SOFTWARE_DATA,  "meta_software.jsonl.gz")
software = pd.read_json(software_path,lines=True, compression="gzip", nrows=10000)
meta_software = pd.read_json(meta_software_path, lines=True, compression="gzip",nrows=10000)
with open(settings_path, 'r') as f:
    cfg = yaml.safe_load(f)

In [10]:
software = pd.DataFrame(software)
meta_software = pd.DataFrame(meta_software)

In [11]:
cfg

{'software': {'text_cols': ['title', 'text'],
  'time_cols': ['timestamp'],
  'num_cols': ['rating'],
  'missing_values': {'drop_is_null': ['user_id'],
   'fill_constant': {'text': 'no_review', 'title': 'no_title'}},
  'numerical': {'nul_cols': 'rating',
   'non_negative': True,
   'min_rate': 1,
   'max_rate': 5}},
 'meta_software': {'text_cols': []}}

In [12]:
software

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1,malware,mcaffee IS malware,[],B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2019-07-03 19:37:12.076,0,False
1,5,Lots of Fun,I love playing tapped out because it is fun to...,[],B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-02-16 20:58:56.000,0,True
2,5,Light Up The Dark,I love this flashlight app! It really illumin...,[],B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2013-03-04 12:14:27.000,0,True
3,4,Fun game,One of my favorite games,[],B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,2019-06-20 20:10:28.662,0,True
4,4,I am not that good at it but my kids are,Cute game. I am not that good at it but my kid...,[],B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,2014-12-11 00:19:56.000,0,True
...,...,...,...,...,...,...,...,...,...,...
9995,5,Alternative to Cutting the Cable Cord,I canceled cable and subscribed to YouTube TV ...,[],B07XF2P9KR,B07XF2P9KR,AENWO4JGIK3IBZYKKTAKJSHU3NNQ,2022-04-13 13:52:54.187,0,True
9996,1,Thumbs down,After being downloaded it wouldn't even open,[],B07856KTQF,B07856KTQF,AEM2STWGDZXSM43OAK55IXMG4BCQ,2022-02-17 01:36:16.105,0,True
9997,5,Having fun,I really like this game. The speed is perfect ...,[],B07JBN7KJ9,B07JBN7KJ9,AEM2STWGDZXSM43OAK55IXMG4BCQ,2021-10-28 21:14:45.187,15,True
9998,4,Love bingo,Really love playing this game. But such a sham...,[],B009WJNXAE,B009WJNXAE,AEM2STWGDZXSM43OAK55IXMG4BCQ,2021-09-30 22:43:02.058,21,True


In [13]:
software.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             10000 non-null  int64         
 1   title              10000 non-null  str           
 2   text               10000 non-null  str           
 3   images             10000 non-null  object        
 4   asin               10000 non-null  str           
 5   parent_asin        10000 non-null  str           
 6   user_id            10000 non-null  str           
 7   timestamp          10000 non-null  datetime64[ms]
 8   helpful_vote       10000 non-null  int64         
 9   verified_purchase  10000 non-null  bool          
dtypes: bool(1), datetime64[ms](1), int64(2), object(1), str(5)
memory usage: 713.0+ KB


In [14]:
software.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='str')

In [15]:
software.describe()

,rating,timestamp,helpful_vote
count,10000.000000,10000,10000.000000
mean,3.848000,2017-06-05 20:47:15.745000,4.118500
min,1.000000,2000-08-08 07:23:31,0.000000
25%,3.000000,2015-05-30 22:09:50,0.000000
50%,5.000000,2017-08-30 04:47:52.939000,0.000000
75%,5.000000,2019-10-01 00:09:51.339000,3.000000
max,5.000000,2023-03-15 15:42:16.874000,797.000000
std,1.459558,NaN,18.591208


In [16]:
software.isnull().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

In [17]:
for col in software.columns:
    print(software[col].value_counts(), "\n")
    print("-----------------------------------")

rating
5    5142
4    1675
1    1385
3    1089
2     709
Name: count, dtype: int64 

-----------------------------------
title
Five Stars                               707
One Star                                 183
Four Stars                               179
Fun                                      137
Three Stars                              118
                                        ... 
Blink Outdoor Security Camera              1
Alternative to Cutting the Cable Cord      1
Thumbs down                                1
Having fun                                 1
Love bingo                                 1
Name: count, Length: 6584, dtype: int64 

-----------------------------------
text
Ok                                                                                                                                                                                                                                                                                                     

In [18]:
software_cfg = cfg["software"]
text_cols = software_cfg.get("text_cols")
text_cols

['title', 'text']

In [19]:
df = software.copy()
df = apply_cleaning_pipeline(df, software_cfg)

In [20]:
df

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,year timestamp,month,day,hour,weekday,quarter,is_weekend,is_month_end,days_since_ref
0,1,malware,mcaffee is malware,[],B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2019-07-03 19:37:12.076,0,False,2019,7,3,19,2,3,False,False,7415
1,5,lots of fun,i love playing tapped out because it is fun to...,[],B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-02-16 20:58:56.000,0,True,2015,2,16,20,0,1,False,False,5817
2,5,light up the dark,i love this flashlight app it really illumina...,[],B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2013-03-04 12:14:27.000,0,True,2013,3,4,12,0,1,False,False,5103
3,4,fun game,one of my favorite games,[],B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,2019-06-20 20:10:28.662,0,True,2019,6,20,20,3,2,False,False,7402
4,4,i am not that good at it but my kids are,cute game i am not that good at it but my kids...,[],B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,2014-12-11 00:19:56.000,0,True,2014,12,11,0,3,4,False,False,5750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,5,alternative to cutting the cable cord,i canceled cable and subscribed to youtube tv ...,[],B07XF2P9KR,B07XF2P9KR,AENWO4JGIK3IBZYKKTAKJSHU3NNQ,2022-04-13 13:52:54.187,0,True,2022,4,13,13,2,2,False,False,8430
9996,1,thumbs down,after being downloaded it wouldnt even open,[],B07856KTQF,B07856KTQF,AEM2STWGDZXSM43OAK55IXMG4BCQ,2022-02-17 01:36:16.105,0,True,2022,2,17,1,3,1,False,False,8375
9997,5,having fun,i really like this game the speed is perfect s...,[],B07JBN7KJ9,B07JBN7KJ9,AEM2STWGDZXSM43OAK55IXMG4BCQ,2021-10-28 21:14:45.187,15,True,2021,10,28,21,3,4,False,False,8263
9998,4,love bingo,really love playing this game but such a shame...,[],B009WJNXAE,B009WJNXAE,AEM2STWGDZXSM43OAK55IXMG4BCQ,2021-09-30 22:43:02.058,21,True,2021,9,30,22,3,3,False,True,8235


# DATA VISUALISATION